In [1]:
import json
import os

from dotenv import load_dotenv

load_dotenv()


True

In [3]:

SAMPLE_WEATHER = {
    "Tokyo": {"celsius": 22, "conditions": "partly cloudy"},
    "Delhi": {"celsius": 34, "conditions": "clear skies"},
    "London": {"celsius": 15, "conditions": "light rain"},
    "Washington, D.C.": {"celsius": 28, "conditions": "sunny"},
    "Paris": {"celsius": 20, "conditions": "overcast"},
}

In [4]:
def get_weather(city: str) -> str:
    data = SAMPLE_WEATHER.get(city.lower())
    if data is None:
        return f"No weather data for {city!r}."
    return f"{city.title()}: {data['celsius']}C, {data['conditions']}"

In [5]:

TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    }
]

TOOLS_BY_NAME = {"get_weather": get_weather}

In [8]:
TOOLS_BY_NAME = {"get_weather": get_weather}

In [7]:

def get_client_and_model():
    """Same provider picker as File 5. Raises if no compatible key is set."""
    from openai import OpenAI

    if os.environ.get("GROQ_API_KEY"):
        return (
            OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1"),
            "llama-3.3-70b-versatile",
        )
    if os.environ.get("OPENROUTER_API_KEY"):
        return (
            OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1"),
            "openrouter/free",
        )
    if os.environ.get("OPENAI_API_KEY"):
        return OpenAI(api_key=os.environ["OPENAI_API_KEY"]), "gpt-4o-mini"

    raise RuntimeError(
        "No OpenAI-compatible key found. Set one of GROQ_API_KEY, "
        "OPENROUTER_API_KEY, or OPENAI_API_KEY in your .env file."
    )

In [9]:

def run_agent(messages: list, max_turns: int = 4) -> str:
    """The agent loop. Each turn: call the model with the tool schema
    attached; if it replies with tool_calls, execute every one of them
    and feed the results back in; if it replies with plain content
    instead, that is the final answer and the loop stops. max_turns is a
    safety limit so a confused model can't loop forever.
    """
    client, model = get_client_and_model()

    for _ in range(max_turns):
        response = client.chat.completions.create(
            model=model, max_tokens=300, messages=messages, tools=TOOL_SCHEMAS
        )
        message = response.choices[0].message

        if not message.tool_calls:
            messages.append({"role": "assistant", "content": message.content})
            return message.content

        messages.append(
            {
                "role": "assistant",
                "content": message.content,
                "tool_calls": [
                    {
                        "id": call.id,
                        "type": "function",
                        "function": {"name": call.function.name, "arguments": call.function.arguments},
                    }
                    for call in message.tool_calls
                ],
            }
        )

In [11]:

def chat() -> None:
    """A minimal terminal REPL. conversation_memory is the agent's entire
    memory -- a plain list, grown by run_agent() on every turn.
    """
    conversation_memory: list[dict] = []
    print("Type a question (Ctrl+C to quit).")

    while True:
        try:
            user_input = input("You: ")
        except (KeyboardInterrupt, EOFError):
            print("\nExiting.")
            break

        conversation_memory.append({"role": "user", "content": user_input})
        answer = run_agent(conversation_memory)
        print(f"Agent: {answer}\n")


In [13]:
if __name__ == "__main__":
    chat()

Type a question (Ctrl+C to quit).
Agent: Hello, I am here to assist you. Is there something I can help you with or would you like to get the current weather for a specific city? If so, please let me know the city.

Agent: 

Agent: 

Agent: 

Agent: 


Exiting.
